In [ ]:
!pip3 install -U neo4j

In [ ]:
from dotenv import load_dotenv
import os
from neo4j import GraphDatabase

In [38]:
# Load the .env file or assign these values manually
load_dotenv()

username = os.getenv("neo4j_username")
password = os.getenv("neo4j_password")

In [82]:
uri = f"neo4j+s://{username}.databases.neo4j.io"

driver = GraphDatabase.driver(uri, auth=(username, password))
driver.verify_connectivity()
session = driver.session()

print("Successfully connected to Neo4j!")

Successfully connected to Neo4j!


Implementing Order by example from : https://neo4j.com/docs/cypher-manual/5/clauses/order-by/

In [24]:
query1 = """
CREATE
    (o1:Order {id: 'ORD-001', orderDate: datetime('2024-05-01T10:00:00'), total: 550, status: 'shipped'}),
    (o2:Order {id: 'ORD-002', orderDate: datetime('2024-05-02T14:30:00'), total: 1000, status: 'pending'}),
    (o3:Order {id: 'ORD-003', orderDate: datetime('2024-05-03T09:15:00'), total: 550, status: 'pending'}),
    (o4:Order {id: 'ORD-004', orderDate: datetime('2024-05-04T12:45:00'), total: 200}),
    (o5:Order {id: 'ORD-005', orderDate: datetime('2024-05-05T15:00:00'), total: 800, status: 'shipped'}),

    (i1:Item {name: 'Phone', price: 500}),
    (i2:Item {name: 'Laptop', price: 1000}),
    (i3:Item {name: 'Headphones', price: 250}),
    (i4:Item {name: 'Charger', price: 50}),
    (i5:Item {name: 'Keyboard', price: 200}),

    (o1)-[:CONTAINS]->(i1),
    (o1)-[:CONTAINS]->(i4),
    (o2)-[:CONTAINS]->(i2),
    (o3)-[:CONTAINS]->(i1),
    (o3)-[:CONTAINS]->(i4),
    (o4)-[:CONTAINS]->(i5),
    (o5)-[:CONTAINS]->(i1),
    (o5)-[:CONTAINS]->(i3),
    (o5)-[:CONTAINS]->(i4)
"""
result1 = session.run(query1)
print("Orders and items created successfully!")

Orders and items created successfully!


In [ ]:
#Order by total order value
query2 = """
MATCH (o:Order)
RETURN o.id AS order,
       o.total AS total
  ORDER BY total
"""
result2 = session.run(query2)
[(record["order"], record["total"]) for record in result2]

[('ORD-004', 200),
 ('ORD-004', 200),
 ('ORD-001', 550),
 ('ORD-003', 550),
 ('ORD-001', 550),
 ('ORD-003', 550),
 ('ORD-005', 800),
 ('ORD-005', 800),
 ('ORD-002', 1000),
 ('ORD-002', 1000)]

In [25]:
query3 = """MATCH (o:Order)
RETURN o.id AS order,
       o.total AS total,
       o.orderDate AS orderDate
  ORDER BY total,
           orderDate"""

result3 = session.run(query3)
[(record["order"], record["total"], record["orderDate"]) for record in result3]

[('ORD-004', 200, neo4j.time.DateTime(2024, 5, 4, 12, 45, 0, 0, tzinfo=<UTC>)),
 ('ORD-001', 550, neo4j.time.DateTime(2024, 5, 1, 10, 0, 0, 0, tzinfo=<UTC>)),
 ('ORD-003', 550, neo4j.time.DateTime(2024, 5, 3, 9, 15, 0, 0, tzinfo=<UTC>)),
 ('ORD-005', 800, neo4j.time.DateTime(2024, 5, 5, 15, 0, 0, 0, tzinfo=<UTC>)),
 ('ORD-002',
  1000,
  neo4j.time.DateTime(2024, 5, 2, 14, 30, 0, 0, tzinfo=<UTC>))]

In [26]:
#This next query counts the number of items contained in each order and then orders the results by the item count.
query4="""MATCH (o:Order)
RETURN o.id AS order,
      COUNT { (o)-[:CONTAINS]->(:Item) } AS itemCount
  ORDER BY itemCount"""

result4 = session.run(query4)
[(record["order"], record["itemCount"]) for record in result4]

[('ORD-002', 1),
 ('ORD-004', 1),
 ('ORD-001', 2),
 ('ORD-003', 2),
 ('ORD-005', 3)]

In [ ]:
#ORDER BY sorts results in an ascending order by default. To explicitly sort results in an ascending order, append ASC[ENDING].
query5="""MATCH (o:Order)
RETURN o.id AS order,
       o.total AS total
  ORDER BY total ASC"""

result5 = session.run(query5)
[(record["order"], record["total"]) for record in result5]

[('ORD-004', 200),
 ('ORD-001', 550),
 ('ORD-003', 550),
 ('ORD-005', 800),
 ('ORD-002', 1000)]

In [53]:
query = ("MATCH (n) DETACH DELETE n")
result = session.run(query)
print("All Nodes and relationships are deleted!")

All Nodes and relationships are deleted!


Kaggle example

In [33]:
!pip3 install kagglehub
!pip3 install "kagglehub[pandas-datasets]" pandas


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
zsh:1: no matches found: kagglehub[pandas-datasets]


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("waqi786/songs-dataset-2000-2020-50k-records")

print("Path to dataset files:", path)

/Users/sd/Documents/APAN5400/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 1.25M/1.25M [00:00<00:00, 6.81MB/s]

Extracting files...


Path to dataset files: /Users/sd/.cache/kagglehub/datasets/waqi786/songs-dataset-2000-2020-50k-records/versions/1


In [ ]:
from pathlib import Path
import pandas as pd

csv_path = Path(path) / "songs_2000_2020_50k.csv"
df = pd.read_csv(csv_path)

print("Loaded:", csv_path)
print("Shape:", df.shape)
df.head()

Loaded: /Users/sd/.cache/kagglehub/datasets/waqi786/songs-dataset-2000-2020-50k-records/versions/1/songs_2000_2020_50k.csv
Shape: (50000, 7)


,Title,Artist,Album,Genre,Release Date,Duration,Popularity
0,Include name this.,Patrick Anderson,Care.,R&B,2008-01-09,262,71
1,Manage west energy.,Eric Miller,Raise get.,Jazz,2011-08-20,187,37
2,Evening court painting.,Richard Curry,Sport.,Electronic,2010-05-30,212,58
3,Section turn hour.,James Smith,Full.,Hip-Hop,2014-10-12,272,59
4,Five agreement teach.,Amy Rodriguez,Eat.,Blues,2005-06-09,131,34


In [56]:
df.groupby('Artist').size().sort_values(ascending=False).head(20)

Artist
James Smith         23
Michael Johnson     21
Robert Johnson      20
Jessica Smith       19
Jennifer Johnson    18
James Brown         16
David Williams      16
Michael Smith       16
David Smith         16
John Smith          15
Jennifer Smith      15
Jessica Williams    15
Nicole Smith        15
Joshua Smith        13
Daniel Johnson      13
Robert Smith        13
Michael Garcia      13
Jessica Johnson     13
Robert Williams     13
John Brown          13
dtype: int64

In [57]:
subset_df = df[df['Artist'].isin(['James Smith', 'Michael Johnson', 'Robert Johnson', 'Jessica Smith'])]
print("Shape:", subset_df.shape)

Shape: (83, 7)


In [78]:
DATA_PATH = Path.cwd().parent / "Module8" / "songs.csv"
subset_df.to_csv(DATA_PATH, index=False)

In [83]:
querysongs = """
LOAD CSV WITH HEADERS FROM 'https://raw.githubusercontent.com/SamarDeen/APAN5400/refs/heads/main/Module8/songs.csv' AS row

// 1. Uniquely create/locate taxonomy and container nodes
MERGE (a:Artist {name: row.Artist})
MERGE (al:Album {name: row.Album})
MERGE (g:Genre {name: row.Genre})
// Merge only the specific song data entry
MERGE (s:Song {title: row.Title})
ON CREATE SET
  s.duration = toInteger(row.Duration),
  s.popularity = toInteger(row.Popularity),
  s.releaseDate = date(row.`Release Date`)

// Connect everything
MERGE (a)-[:SINGS]->(s)-[:IN_ALBUM]->(al)
MERGE (s)-[:HAS_GENRE]->(g)
"""

result_query_songs = session.run(querysongs)


Now go to neo4j browser and see the full network by running command:
MATCH (n) RETURN n

Now let's try to see which is the most popular genre
MATCH (s:Song)-[:HAS_GENRE]->(g:Genre)
RETURN g.name AS genre, sum(s.popularity) AS total_popularity
ORDER BY total_popularity DESC
LIMIT 2

In [84]:
query = ("MATCH (n) DETACH DELETE n")
result = session.run(query)
print("All Nodes and relationships are deleted!")

All Nodes and relationships are deleted!


In [85]:
session.close()
driver.close()